# GO-associated structure in ESM-1b embeddings

This notebook tests whether ESM-1b embeddings show unsupervised structure associated with direct experimental GO annotations. It creates one molecular-function, biological-process, and cellular-component analysis for both E. coli and human.

Each analysis uses three pairwise-disjoint GO groups with exactly 20 proteins per group. Labels are used only for sampling, coloring, and post-hoc validation; UMAP and t-SNE are fitted without GO labels.

The final section extends the analysis to a balanced annotated-versus-unannotated comparison, including cosine-similarity distributions and jointly fitted 3D PCA, UMAP, and t-SNE projections for the interactive explorer.

In [1]:
from pathlib import Path
from collections import defaultdict
import configparser
import hashlib
import json
import os
import re
import sys

os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/numba-s2f-esm-go')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-s2f-esm-go')
Path(os.environ['NUMBA_CACHE_DIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances, silhouette_score
import umap

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'plm.py').exists():
    raise FileNotFoundError(f'Run from the S2F repository or notebooks directory: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))
import plm

CONFIG_PATH = PROJECT_ROOT / 's2f.conf'
config = configparser.ConfigParser()
config.read(CONFIG_PATH)
TRANSFER_DATA = PROJECT_ROOT / 'transfer' / 'esm_pfp_work_2026-07-01' / 'data'


def first_existing_path(*candidates):
    for candidate in candidates:
        path = Path(candidate).expanduser()
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f'None of the expected paths exists: {candidates}')


FILTERED_GOA = first_existing_path(
    config.get('databases', 'filtered_goa'),
    TRANSFER_DATA / 'uniprot' / 'filtered_goa',
)
SOURCE_GOA = Path(config.get('databases', 'uniprot_goa')).expanduser()
ECOLI_CACHE = first_existing_path(
    '/media/marcelo_baez/HD_Disc1/.S2F/output/83333/83333_PLM/query_embeddings',
    TRANSFER_DATA / 'embeddings' / 'ecoli_83333_query_embeddings',
)
HUMAN_CACHE = first_existing_path(
    '/media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot',
    TRANSFER_DATA / 'embeddings' / 'human_9606_all_swissprot',
)
FRONTEND_DIR = PROJECT_ROOT / 'notebooks' / 'esm_go_explorer'
DATA_DIR = FRONTEND_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_CODES = {
    value.strip()
    for value in config.get('options', 'evidence_codes').split(',')
    if value.strip()
}
RANDOM_SEED = 42
GROUP_SIZE = 20
PERMUTATIONS = 10_000

print('Python:', sys.executable)
print('NumPy:', np.__version__, 'pandas:', pd.__version__, 'UMAP:', umap.__version__)
print('GOA:', FILTERED_GOA)
print('E. coli cache:', ECOLI_CACHE)
print('Human cache:', HUMAN_CACHE)
print('Frontend data:', DATA_DIR)

Python: /home/marcelo_baez/anaconda3/envs/S2F/bin/python
NumPy: 2.4.2 pandas: 3.0.1 UMAP: 0.5.12
GOA: /media/marcelo_baez/HD_Disc1/.S2F/data/UniprotKB/filtered_goa
E. coli cache: /media/marcelo_baez/HD_Disc1/.S2F/output/83333/83333_PLM/query_embeddings
Human cache: /media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot
Frontend data: /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data


## Fixed GO groups and colors

In [2]:
ORGANISMS = {
    'ecoli': {'taxon': '83333', 'name': 'Escherichia coli', 'cache': ECOLI_CACHE},
    'human': {'taxon': '9606', 'name': 'Homo sapiens', 'cache': HUMAN_CACHE},
}

GO_GROUPS = {
    'MF': [
        {'go_id': 'GO:0009055', 'name': 'electron transfer activity', 'color': '#0ea5e9'},
        {'go_id': 'GO:0000987', 'name': 'cis-regulatory region sequence-specific DNA binding', 'color': '#f97316'},
        {'go_id': 'GO:0046872', 'name': 'metal ion binding', 'color': '#8b5cf6'},
    ],
    'BP': [
        {'go_id': 'GO:0006281', 'name': 'DNA repair', 'color': '#14b8a6'},
        {'go_id': 'GO:0006351', 'name': 'DNA-templated transcription', 'color': '#ec4899'},
        {'go_id': 'GO:0051289', 'name': 'protein homotetramerization', 'color': '#6366f1'},
    ],
    'CC': [
        {'go_id': 'GO:0032993', 'name': 'protein-DNA complex', 'color': '#10b981'},
        {'go_id': 'GO:0022625', 'name': 'cytosolic large ribosomal subunit', 'color': '#ef4444'},
        {'go_id': 'GO:0022627', 'name': 'cytosolic small ribosomal subunit', 'color': '#eab308'},
    ],
}

DOMAIN_NAMES = {
    'MF': 'Molecular Function',
    'BP': 'Biological Process',
    'CC': 'Cellular Component',
}

pd.DataFrame([
    {'domain': domain, 'go_id': group['go_id'], 'go_name': group['name'], 'color': group['color']}
    for domain, groups in GO_GROUPS.items()
    for group in groups
])

Out[1]: 
  domain  ...    color
0     MF  ...  #0ea5e9
1     MF  ...  #f97316
2     MF  ...  #8b5cf6
3     BP  ...  #14b8a6
4     BP  ...  #ec4899
5     BP  ...  #6366f1
6     CC  ...  #10b981
7     CC  ...  #ef4444
8     CC  ...  #eab308

[9 rows x 4 columns]


## Load embedding caches and direct annotations

Only direct GOA rows are used. `NOT` qualifiers are excluded, evidence codes are filtered explicitly, and annotations are intersected with each organism's embedding cache before group validation.

In [3]:
caches = {}
cache_indices = {}
cache_lengths = {}
for organism_key, spec in ORGANISMS.items():
    cache = plm.load_embedding_cache(spec['cache'], validate=False)
    if cache is None:
        raise RuntimeError(f'Incomplete embedding cache: {spec["cache"]}')
    if cache.embeddings.shape[1] != 1280:
        raise ValueError(f'Unexpected embedding dimension for {organism_key}: {cache.embeddings.shape}')
    caches[organism_key] = cache
    cache_indices[organism_key] = {protein_id: index for index, protein_id in enumerate(cache.ids)}
    cache_lengths[organism_key] = dict(zip(cache.ids, cache.lengths))

cache_summary = pd.DataFrame([
    {
        'organism': ORGANISMS[key]['name'],
        'taxon': ORGANISMS[key]['taxon'],
        'proteins': len(cache.ids),
        'embedding_dimension': cache.embeddings.shape[1],
        'model': cache.metadata.get('model_name'),
        'long_sequence_mode': cache.metadata.get('long_sequence_mode'),
    }
    for key, cache in caches.items()
])
display(cache_summary)

selected_go_ids = {group['go_id'] for groups in GO_GROUPS.values() for group in groups}
goa_columns = [
    'DB', 'Protein', 'Symbol', 'Qualifier', 'GO', 'Reference', 'Evidence', 'With',
    'Aspect', 'ObjectName', 'Synonym', 'ObjectType', 'Taxon', 'Date', 'AssignedBy',
    'AnnotationExtension', 'GeneProductForm',
]
annotations = {key: defaultdict(set) for key in ORGANISMS}

for chunk in pd.read_csv(
    FILTERED_GOA,
    sep='\t',
    header=None,
    names=goa_columns,
    dtype=str,
    chunksize=500_000,
    low_memory=False,
):
    chunk = chunk[chunk['GO'].isin(selected_go_ids) & chunk['Evidence'].isin(EVIDENCE_CODES)]
    if chunk.empty:
        continue
    chunk = chunk[~chunk['Qualifier'].fillna('').str.contains(r'(?:^|\|)NOT(?:$|\|)', regex=True)]
    for organism_key, spec in ORGANISMS.items():
        taxon_pattern = fr'(?:^|\|)taxon:{spec["taxon"]}(?:$|\|)'
        subset = chunk[
            chunk['Taxon'].fillna('').str.contains(taxon_pattern, regex=True)
            & chunk['Protein'].isin(cache_indices[organism_key])
        ]
        for protein_id, go_id in subset[['Protein', 'GO']].itertuples(index=False, name=None):
            annotations[organism_key][go_id].add(protein_id)

eligible_rows = []
for organism_key, spec in ORGANISMS.items():
    for domain, groups in GO_GROUPS.items():
        group_sets = [annotations[organism_key][group['go_id']] for group in groups]
        overlaps = [
            group_sets[i] & group_sets[j]
            for i in range(3)
            for j in range(i + 1, 3)
        ]
        if any(overlaps):
            raise ValueError(f'GO groups overlap for {organism_key} {domain}: {[len(x) for x in overlaps]}')
        for group, proteins in zip(groups, group_sets):
            if len(proteins) < GROUP_SIZE:
                raise ValueError(f'{organism_key} {group["go_id"]} has only {len(proteins)} eligible proteins.')
            eligible_rows.append({
                'organism': spec['name'],
                'taxon': spec['taxon'],
                'domain': domain,
                'go_id': group['go_id'],
                'go_name': group['name'],
                'eligible_proteins': len(proteins),
            })

ELIGIBLE_COUNTS_DF = pd.DataFrame(eligible_rows)
display(ELIGIBLE_COUNTS_DF)

,organism,taxon,proteins,embedding_dimension,model,long_sequence_mode
0,Escherichia coli,83333,16075,1280,facebook/esm1b_t33_650M_UR50S,sliding_mean
1,Homo sapiens,9606,20420,1280,facebook/esm1b_t33_650M_UR50S,sliding_mean


,organism,taxon,domain,go_id,go_name,eligible_proteins
0,Escherichia coli,83333,MF,GO:0009055,electron transfer activity,37
1,Escherichia coli,83333,MF,GO:0000987,cis-regulatory region sequence-specific DNA bi...,29
2,Escherichia coli,83333,MF,GO:0046872,metal ion binding,37
3,Escherichia coli,83333,BP,GO:0006281,DNA repair,22
4,Escherichia coli,83333,BP,GO:0006351,DNA-templated transcription,53
5,Escherichia coli,83333,BP,GO:0051289,protein homotetramerization,47
6,Escherichia coli,83333,CC,GO:0032993,protein-DNA complex,25
7,Escherichia coli,83333,CC,GO:0022625,cytosolic large ribosomal subunit,34
8,Escherichia coli,83333,CC,GO:0022627,cytosolic small ribosomal subunit,23
9,Homo sapiens,9606,MF,GO:0009055,electron transfer activity,45


## Select balanced groups and calculate distances, projections, and validation

In [4]:
def stable_seed(*parts):
    digest = hashlib.sha256('|'.join(map(str, parts)).encode('utf-8')).hexdigest()
    return (RANDOM_SEED + int(digest[:8], 16)) % (2**32)


def sample_group(proteins, organism_key, domain, go_id):
    candidates = np.array(sorted(proteins), dtype=object)
    rng = np.random.default_rng(stable_seed(organism_key, domain, go_id))
    selected = rng.choice(candidates, size=GROUP_SIZE, replace=False)
    return sorted(selected.tolist())


def matrix_validation_statistics(matrix, labels, larger_is_closer, permutations=PERMUTATIONS):
    row_indices, col_indices = np.triu_indices(matrix.shape[0], k=1)
    pair_values = matrix[row_indices, col_indices]
    within_mask = labels[row_indices] == labels[col_indices]
    within = pair_values[within_mask]
    between = pair_values[~within_mask]
    if larger_is_closer:
        observed = float(within.mean() - between.mean())
        separation_direction = 'within_minus_between'
    else:
        observed = float(between.mean() - within.mean())
        separation_direction = 'between_minus_within'

    rng = np.random.default_rng(RANDOM_SEED)
    null_statistics = np.empty(permutations, dtype=np.float32)
    batch_size = 500
    for start in range(0, permutations, batch_size):
        end = min(start + batch_size, permutations)
        shuffled = np.stack([rng.permutation(labels) for _ in range(end - start)])
        shuffled_within = shuffled[:, row_indices] == shuffled[:, col_indices]
        within_counts = shuffled_within.sum(axis=1)
        between_counts = pair_values.size - within_counts
        within_means = (shuffled_within * pair_values).sum(axis=1) / within_counts
        between_means = ((~shuffled_within) * pair_values).sum(axis=1) / between_counts
        if larger_is_closer:
            null_statistics[start:end] = within_means - between_means
        else:
            null_statistics[start:end] = between_means - within_means
    p_value = float((1 + np.count_nonzero(null_statistics >= observed)) / (permutations + 1))
    return {
        'within_mean': float(within.mean()),
        'within_median': float(np.median(within)),
        'within_std': float(within.std()),
        'within_pairs': int(within.size),
        'between_mean': float(between.mean()),
        'between_median': float(np.median(between)),
        'between_std': float(between.std()),
        'between_pairs': int(between.size),
        'separation_mean_difference': observed,
        'separation_direction': separation_direction,
        'permutation_p_value': p_value,
        'permutations': permutations,
    }


MATRIX_METRICS = {
    'cosine_similarity': {
        'key': 'cosine_similarity',
        'label': 'Cosine similarity',
        'kind': 'similarity',
        'value_label': 'Cosine similarity',
        'range_min_label': 'lower similarity',
        'range_max_label': 'higher similarity',
        'larger_is_closer': True,
        'diagonal_value': 1.0,
        'description': 'Higher values mean the embeddings point in more similar directions.',
    },
    'euclidean_distance': {
        'key': 'euclidean_distance',
        'label': 'Euclidean distance',
        'kind': 'distance',
        'value_label': 'Euclidean distance',
        'range_min_label': 'lower distance',
        'range_max_label': 'higher distance',
        'larger_is_closer': False,
        'diagonal_value': 0.0,
        'description': 'Lower values mean the embeddings are closer in 1,280-dimensional space.',
    },
}
DEFAULT_MATRIX_METRIC = 'cosine_similarity'


analysis_results = {}
manifest_rows = []
summary_rows = []

for organism_key, spec in ORGANISMS.items():
    cache = caches[organism_key]
    for domain, groups in GO_GROUPS.items():
        ordered_ids = []
        labels = []
        group_metadata = []
        for group_index, group in enumerate(groups):
            chosen = sample_group(
                annotations[organism_key][group['go_id']],
                organism_key,
                domain,
                group['go_id'],
            )
            start = len(ordered_ids)
            ordered_ids.extend(chosen)
            labels.extend([group_index] * len(chosen))
            group_metadata.append({
                **group,
                'group_index': group_index,
                'start': start,
                'end': start + len(chosen),
                'eligible_count': len(annotations[organism_key][group['go_id']]),
                'selected_count': len(chosen),
            })
            for protein_id in chosen:
                manifest_rows.append({
                    'organism_key': organism_key,
                    'organism': spec['name'],
                    'taxon': spec['taxon'],
                    'domain': domain,
                    'group_index': group_index,
                    'go_id': group['go_id'],
                    'go_name': group['name'],
                    'color': group['color'],
                    'protein_id': protein_id,
                    'sequence_length': cache_lengths[organism_key][protein_id],
                })

        labels_array = np.asarray(labels, dtype=int)
        positions = [cache_indices[organism_key][protein_id] for protein_id in ordered_ids]
        embeddings = np.asarray(cache.embeddings[positions], dtype=np.float32)

        cosine_distance_matrix = pairwise_distances(embeddings, metric='cosine').astype(np.float32)
        cosine_distance_matrix = np.clip(cosine_distance_matrix, 0.0, 2.0)
        np.fill_diagonal(cosine_distance_matrix, 0.0)

        cosine_similarity_matrix = (1.0 - cosine_distance_matrix).astype(np.float32)
        cosine_similarity_matrix = np.clip(cosine_similarity_matrix, -1.0, 1.0)
        np.fill_diagonal(cosine_similarity_matrix, 1.0)

        euclidean_distance_matrix = pairwise_distances(embeddings, metric='euclidean').astype(np.float32)
        np.fill_diagonal(euclidean_distance_matrix, 0.0)

        umap_coordinates = umap.UMAP(
            n_neighbors=10,
            min_dist=0.1,
            n_components=2,
            metric='cosine',
            random_state=RANDOM_SEED,
            n_jobs=1,
        ).fit_transform(embeddings).astype(np.float32)
        tsne_coordinates = TSNE(
            n_components=2,
            perplexity=15,
            metric='cosine',
            init='random',
            learning_rate='auto',
            max_iter=1500,
            random_state=RANDOM_SEED,
        ).fit_transform(embeddings).astype(np.float32)

        matrix_arrays = {
            'cosine_similarity': cosine_similarity_matrix,
            'euclidean_distance': euclidean_distance_matrix,
        }
        silhouette_distances = {
            'cosine_similarity': cosine_distance_matrix,
            'euclidean_distance': euclidean_distance_matrix,
        }
        matrices = {}
        metric_statistics = {}
        for metric_key, metric_info in MATRIX_METRICS.items():
            matrix = matrix_arrays[metric_key]
            stats = matrix_validation_statistics(
                matrix,
                labels_array,
                larger_is_closer=metric_info['larger_is_closer'],
            )
            stats['metric_key'] = metric_key
            stats['metric_label'] = metric_info['label']
            stats['metric_kind'] = metric_info['kind']
            stats['silhouette'] = float(
                silhouette_score(silhouette_distances[metric_key], labels_array, metric='precomputed')
            )
            stats['protein_count'] = len(ordered_ids)
            metric_statistics[metric_key] = stats
            matrices[metric_key] = {
                **{key: value for key, value in metric_info.items() if key != 'larger_is_closer'},
                'matrix': np.round(matrix, 8).tolist(),
                'statistics': stats,
            }

        proteins = []
        for index, (protein_id, group_index) in enumerate(zip(ordered_ids, labels)):
            group = groups[group_index]
            proteins.append({
                'index': index,
                'protein_id': protein_id,
                'sequence_length': int(cache_lengths[organism_key][protein_id]),
                'group_index': group_index,
                'go_id': group['go_id'],
                'go_name': group['name'],
                'color': group['color'],
            })

        dataset_id = f'{organism_key}_{domain.lower()}'
        default_statistics = metric_statistics[DEFAULT_MATRIX_METRIC]
        result = {
            'schema_version': 2,
            'metadata': {
                'dataset_id': dataset_id,
                'organism_key': organism_key,
                'organism': spec['name'],
                'taxon': spec['taxon'],
                'domain': domain,
                'domain_name': DOMAIN_NAMES[domain],
                'model_name': cache.metadata.get('model_name'),
                'embedding_dimension': 1280,
                'pooling': 'residue mean; sliding-window mean for long sequences',
                'default_matrix_metric': DEFAULT_MATRIX_METRIC,
                'matrix_metrics': list(MATRIX_METRICS.keys()),
                'projection_metric': 'cosine',
                'random_seed': RANDOM_SEED,
                'group_size': GROUP_SIZE,
                'direct_annotations_only': True,
                'evidence_codes': sorted(EVIDENCE_CODES),
            },
            'groups': group_metadata,
            'proteins': proteins,
            'matrix_metrics': [
                {key: value for key, value in metric_info.items() if key != 'larger_is_closer'}
                for metric_info in MATRIX_METRICS.values()
            ],
            'matrices': matrices,
            'similarity_matrix': np.round(cosine_similarity_matrix, 8).tolist(),
            'distance_matrix': np.round(cosine_distance_matrix, 8).tolist(),
            'umap': np.round(umap_coordinates, 8).tolist(),
            'tsne': np.round(tsne_coordinates, 8).tolist(),
            'statistics': {
                **default_statistics,
                'silhouette_cosine': default_statistics['silhouette'],
            },
        }
        analysis_results[dataset_id] = result
        for metric_key, stats in metric_statistics.items():
            summary_rows.append({
                'dataset_id': dataset_id,
                'organism': spec['name'],
                'taxon': spec['taxon'],
                'domain': domain,
                **stats,
            })
        print(
            f'Completed {dataset_id}: '
            f'cosine_similarity silhouette={metric_statistics["cosine_similarity"]["silhouette"]:.4f}, '
            f'euclidean_distance silhouette={metric_statistics["euclidean_distance"]["silhouette"]:.4f}'
        )


Completed ecoli_mf: cosine_similarity silhouette=0.0771, euclidean_distance silhouette=0.0448
Completed ecoli_bp: cosine_similarity silhouette=0.0191, euclidean_distance silhouette=0.0152
Completed ecoli_cc: cosine_similarity silhouette=0.0683, euclidean_distance silhouette=0.0370
Completed human_mf: cosine_similarity silhouette=0.0778, euclidean_distance silhouette=0.0491
Completed human_bp: cosine_similarity silhouette=-0.0121, euclidean_distance silhouette=0.0010
Completed human_cc: cosine_similarity silhouette=0.0782, euclidean_distance silhouette=0.0471


## Export frontend data and inspect results

In [5]:
MANIFEST_DF = pd.DataFrame(manifest_rows)
SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(
    ['organism', 'domain', 'metric_key']
).reset_index(drop=True)


dataset_index = []
for dataset_id, result in analysis_results.items():
    output_path = DATA_DIR / f'{dataset_id}.json'
    with output_path.open('w', encoding='utf-8') as handle:
        json.dump(result, handle, separators=(',', ':'), allow_nan=False)
    dataset_index.append({
        'dataset_id': dataset_id,
        'organism_key': result['metadata']['organism_key'],
        'organism': result['metadata']['organism'],
        'taxon': result['metadata']['taxon'],
        'domain': result['metadata']['domain'],
        'domain_name': result['metadata']['domain_name'],
        'path': f'data/{dataset_id}.json',
        'default_matrix_metric': result['metadata']['default_matrix_metric'],
        'matrix_metrics': result['metadata']['matrix_metrics'],
    })

goa_provenance = {}
if SOURCE_GOA.exists():
    with SOURCE_GOA.open('r', encoding='utf-8', errors='replace') as handle:
        for _ in range(100):
            line = handle.readline()
            if not line.startswith('!'):
                break
            if ':' in line:
                key, value = line[1:].strip().split(':', 1)
                goa_provenance[key.strip()] = value.strip()

index_payload = {
    'schema_version': 2,
    'default_dataset': 'ecoli_mf',
    'default_matrix_metric': DEFAULT_MATRIX_METRIC,
    'matrix_metrics': [
        {key: value for key, value in metric_info.items() if key != 'larger_is_closer'}
        for metric_info in MATRIX_METRICS.values()
    ],
    'datasets': dataset_index,
    'goa_provenance': goa_provenance,
    'scientific_scope': (
        'These unsupervised similarities, distances, and projections test GO-associated structure in ESM-1b embeddings. '
        'They do not establish that GO labels were memorized during pretraining or identify causal dimensions.'
    ),
}
with (DATA_DIR / 'index.json').open('w', encoding='utf-8') as handle:
    json.dump(index_payload, handle, indent=2, sort_keys=True, allow_nan=False)

MANIFEST_DF.to_csv(DATA_DIR / 'selected_proteins.csv', index=False)
SUMMARY_DF.to_csv(DATA_DIR / 'summary_statistics.csv', index=False)

for dataset_id, result in analysis_results.items():
    assert result['schema_version'] == 2
    cosine_similarity = np.asarray(result['matrices']['cosine_similarity']['matrix'], dtype=float)
    cosine_distance = np.asarray(result['distance_matrix'], dtype=float)
    euclidean_distance = np.asarray(result['matrices']['euclidean_distance']['matrix'], dtype=float)
    for matrix in [cosine_similarity, cosine_distance, euclidean_distance]:
        assert matrix.shape == (60, 60)
        assert np.all(np.isfinite(matrix))
        assert np.allclose(matrix, matrix.T, atol=1e-6)
    assert np.allclose(np.diag(cosine_similarity), 1.0, atol=1e-8)
    assert np.allclose(np.diag(cosine_distance), 0.0, atol=1e-8)
    assert np.allclose(np.diag(euclidean_distance), 0.0, atol=1e-8)
    assert np.allclose(cosine_similarity, 1.0 - cosine_distance, atol=1e-6)
    assert [group['start'] for group in result['groups']] == [0, 20, 40]
    assert [group['end'] for group in result['groups']] == [20, 40, 60]
    assert np.asarray(result['umap']).shape == (60, 2)
    assert np.asarray(result['tsne']).shape == (60, 2)

summary_display = SUMMARY_DF[[
    'organism', 'domain', 'metric_label', 'within_mean', 'between_mean',
    'separation_mean_difference', 'separation_direction', 'silhouette', 'permutation_p_value',
]]
display(summary_display)
display(MANIFEST_DF.groupby(['organism', 'domain', 'go_id', 'go_name'], as_index=False).size())
print(f'Wrote {len(analysis_results)} interactive datasets to {DATA_DIR.resolve()}')


Wrote 6 interactive datasets to /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data


,organism,domain,metric_label,within_mean,between_mean,separation_mean_difference,separation_direction,silhouette,permutation_p_value
0,Escherichia coli,BP,Cosine similarity,0.972848,0.970883,0.001965,within_minus_between,0.019141,0.0001
1,Escherichia coli,BP,Euclidean distance,4.134911,4.300253,0.165342,between_minus_within,0.015156,0.0001
2,Escherichia coli,CC,Cosine similarity,0.973680,0.968661,0.005019,within_minus_between,0.068342,0.0001
3,Escherichia coli,CC,Euclidean distance,4.126999,4.512105,0.385105,between_minus_within,0.037001,0.0001
4,Escherichia coli,MF,Cosine similarity,0.975679,0.972167,0.003512,within_minus_between,0.077131,0.0001
5,Escherichia coli,MF,Euclidean distance,3.934702,4.235269,0.300567,between_minus_within,0.044768,0.0001
6,Homo sapiens,BP,Cosine similarity,0.963033,0.960390,0.002643,within_minus_between,-0.012125,0.0003
7,Homo sapiens,BP,Euclidean distance,4.751236,4.947670,0.196434,between_minus_within,0.000962,0.0001
8,Homo sapiens,CC,Cosine similarity,0.967561,0.958364,0.009197,within_minus_between,0.078242,0.0001
9,Homo sapiens,CC,Euclidean distance,4.543267,5.150493,0.607226,between_minus_within,0.047106,0.0001


,organism,domain,go_id,go_name,size
0,Escherichia coli,BP,GO:0006281,DNA repair,20
1,Escherichia coli,BP,GO:0006351,DNA-templated transcription,20
2,Escherichia coli,BP,GO:0051289,protein homotetramerization,20
3,Escherichia coli,CC,GO:0022625,cytosolic large ribosomal subunit,20
4,Escherichia coli,CC,GO:0022627,cytosolic small ribosomal subunit,20
5,Escherichia coli,CC,GO:0032993,protein-DNA complex,20
6,Escherichia coli,MF,GO:0000987,cis-regulatory region sequence-specific DNA bi...,20
7,Escherichia coli,MF,GO:0009055,electron transfer activity,20
8,Escherichia coli,MF,GO:0046872,metal ion binding,20
9,Homo sapiens,BP,GO:0006281,DNA repair,20


## Annotated versus unannotated proteins

This extension keeps the existing 60 annotated proteins for each organism/domain dataset and selects 60 proteins with no accepted direct experimental GO row across MF, BP, or CC. Unannotated proteins are matched without replacement to the annotated sample by nearest sequence length. This definition is intentionally scoped to the filtered GOA snapshot and configured evidence codes; it does not claim absence from every GOA evidence class or a future release.

The generator L2-normalizes the ESM embeddings, computes annotated/annotated, unannotated/unannotated, and annotated/unannotated cosine distributions, and fits PCA, 3D UMAP, and 3D t-SNE jointly on the balanced 120-protein sample. It exports the comparison statistics, selected-protein manifest, projection coordinates, hover metadata, and explorer routing.

In [ ]:
import subprocess

annotation_env = os.environ.copy()
annotation_env.setdefault('NUMBA_CACHE_DIR', '/tmp/numba-s2f-annotation-latent')
annotation_env.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-s2f-annotation-latent')
annotation_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'build_annotation_latent_analysis.py'),
    '--project-root', str(PROJECT_ROOT),
    '--sync-transfer',
]
completed = subprocess.run(
    annotation_command,
    cwd=PROJECT_ROOT,
    env=annotation_env,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)
ANNOTATION_STATUS_SUMMARY_DF = pd.read_csv(DATA_DIR / 'annotation_status_summary.csv')
display(ANNOTATION_STATUS_SUMMARY_DF[[
    'organism', 'domain', 'comparison', 'pairs', 'mean', 'median', 'std',
    'separation_mean_difference', 'silhouette_cosine', 'permutation_p_value',
]])


## Interpretation guide

- Cosine similarity has diagonal values of 1.0 because each protein is maximally similar to itself.
- Euclidean distance has diagonal values of 0.0 because each protein has zero distance to itself.
- Positive `separation_mean_difference` always means stronger GO-group separation: `within_minus_between` for similarity metrics and `between_minus_within` for distance metrics.
- Positive silhouette values indicate that proteins tend to be closer to their assigned GO group than to other groups.
- The permutation p-value asks whether the observed within/between separation is stronger than random balanced labels.
- UMAP and t-SNE are exploratory nonlinear views fitted with cosine geometry; apparent islands must be interpreted together with the matrix-specific statistics.
- In the annotation-status extension, a positive within-status minus between-status difference indicates separation, but its magnitude and cosine silhouette must be read alongside the permutation p-value. Statistical detectability alone does not imply a large biological effect.
- PCA, 3D UMAP, and 3D t-SNE are fitted jointly on annotated and unannotated proteins; the coordinates are therefore directly comparable within each organism/domain dataset.
- This analysis concerns whole embeddings. It does not test which individual dimensions carry GO-associated information.
